# laya RLCD training on Kaggle (parameterized)

Headless via the push->poll->pull pattern (see docs/training-plan.md 6.1):

```bash
kaggle kernels push -p cloud/kaggle   # upload triggers a GPU run
kaggle kernels status <user>/train    # poll until complete
kaggle kernels output <user>/train -p out/
```

or with kagglepipe (github.com/AdityaProCoder/kagglepipe): one command
wraps render + push + poll + output pull.

Set the accelerator to T4x2 (Settings -> Accelerator). The checkpoint is
pushed to HF Hub; the report lands in /kaggle/working (pulled by
`kernels output`).

In [ ]:
EPOCHS = 4
UNFREEZE = False
HF_REPO = "hathawayandrx105/laya-canon-head"

In [ ]:
import subprocess

subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/NandhaKishorM/laya"], check=True)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/hathawayANdRX105/laya-canon-judgments"],
               check=True)
subprocess.run(["pip", "install", "-e", "/kaggle/working/laya"], check=True)

In [ ]:
from laya.train import TrainConfig, finetune
from laya.agent import _fix_tokenizer_config

model_dir = "convaiinnovations/laya"
_fix_tokenizer_config(model_dir)
config = TrainConfig(
    epochs=EPOCHS, micro_batch=4, grad_accum=2,
    freeze_encoder=not UNFREEZE, seed=42, calib_seed=42,
    max_len=1024, head_max_len=256,
)
summary = finetune(
    data="/kaggle/working/laya-canon-judgments/data/splits/train.jsonl",
    model_dir=model_dir,
    output_dir="/kaggle/working/out",
    config=config,
    device="cuda",
)
print(summary)

In [ ]:
import json
import os

from huggingface_hub import HfApi

with open("/kaggle/working/report.json", "w") as f:
    json.dump({"summary": str(summary), "epochs": EPOCHS,
               "unfreeze": UNFREEZE}, f, indent=2)

api = HfApi(token=os.environ.get("HF_TOKEN"))
if api:
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, folder_path="/kaggle/working/out",
                      path_in_repo="")
    print("checkpoint pushed to", HF_REPO)
else:
    print("HF_TOKEN not set; checkpoint stays in /kaggle/working/out")